In [1]:
from collections import Counter

# Training corpus
corpus = [
    "<s> I love NLP </s>",
    "<s> I love deep learning </s>",
    "<s> deep learning is fun </s>",
]

def tok(s):
    return s.split()

# Unigram & bigram counts
unigrams = Counter()
bigrams = Counter()
context_totals = Counter()   # total next-words seen after a given token

for line in corpus:
    w = tok(line)
    unigrams.update(w)
    for i in range(len(w) - 1):
        prev_, cur = w[i], w[i+1]
        bigrams[(prev_, cur)] += 1
        context_totals[prev_] += 1

def bigram_mle(prev_, cur):
    """ P(cur | prev) by MLE """
    num = bigrams.get((prev_, cur), 0)
    den = context_totals.get(prev_, 0)
    return (num / den) if den > 0 else 0.0

def sentence_prob(sentence):
    """ Product of bigram MLEs over the sentence """
    w = tok(sentence)
    p = 1.0
    steps = []
    for i in range(len(w) - 1):
        prev_, cur = w[i], w[i+1]
        pc = bigram_mle(prev_, cur)
        steps.append((prev_, cur, pc))
        p *= pc
    return p, steps

# Test sentences
S1 = "<s> I love NLP </s>"
S2 = "<s> I love deep learning </s>"

p1, steps1 = sentence_prob(S1)
p2, steps2 = sentence_prob(S2)

# ---- Print results ----
print("=== Unigram Counts ===")
for w, c in sorted(unigrams.items()):
    print(f"{w:>10}: {c}")

print("\n=== Bigram Counts (prev -> next) ===")
for (a, b), c in sorted(bigrams.items()):
    print(f"{a:>10} -> {b:<10}: {c}")

print("\n=== S1 steps ===")
for a, b, pc in steps1:
    print(f"P({b} | {a}) = {pc:.4f}")
print(f"S1 Probability = {p1:.6f}")

print("\n=== S2 steps ===")
for a, b, pc in steps2:
    print(f"P({b} | {a}) = {pc:.4f}")
print(f"S2 Probability = {p2:.6f}")

print("\nPreferred:", "S1" if p1 > p2 else "S2" if p2 > p1 else "Tie")
if p1 != p2:
    print("Why:", "Higher product of bigram probabilities (MLE).")


=== Unigram Counts ===
      </s>: 3
       <s>: 3
         I: 2
       NLP: 1
      deep: 2
       fun: 1
        is: 1
  learning: 2
      love: 2

=== Bigram Counts (prev -> next) ===
       <s> -> I         : 2
       <s> -> deep      : 1
         I -> love      : 2
       NLP -> </s>      : 1
      deep -> learning  : 2
       fun -> </s>      : 1
        is -> fun       : 1
  learning -> </s>      : 1
  learning -> is        : 1
      love -> NLP       : 1
      love -> deep      : 1

=== S1 steps ===
P(I | <s>) = 0.6667
P(love | I) = 1.0000
P(NLP | love) = 0.5000
P(</s> | NLP) = 1.0000
S1 Probability = 0.333333

=== S2 steps ===
P(I | <s>) = 0.6667
P(love | I) = 1.0000
P(deep | love) = 0.5000
P(learning | deep) = 1.0000
P(</s> | learning) = 0.5000
S2 Probability = 0.166667

Preferred: S1
Why: Higher product of bigram probabilities (MLE).
